# Universidad Tecmilenio
## Curso: Desarrollo de Aplicaciones con IA
### Actividad: Avance de Proyecto 1
**Asistente Inteligente para la Inspección Visual y Trazabilidad de Daños en Empaques durante la Recepción de Mercancías**

### Contexto de Negocio y Dominio del Proyecto
- **Propósito:** Asistir a inspectores de almacén en andenes de carga para evaluar paquetes (empaques intactos vs. dañados).
- **Alcance actual (Avance 1):** Chatbot conversacional basado en reglas y expresiones regulares desarrollado en Python con **NLTK** (tokenización y stemming en español), extracción de identificadores operativos (lote y andén), clasificación de intenciones por coincidencia de palabras/frases clave (`keywords`/`frases_clave`), gestión de sesión con llenado de datos (*slot-filling*), emisión de respuestas y fallbacks controlados, y bitácora transaccional.

- **Evolución posterior (Avances 2 y Proyecto Final):** Se integrará un clasificador supervisado con **TF-IDF + SVM** para clasificación de intenciones, una CNN binaria (**ResNet50**) para clasificación visual de empaques (`caja_intacta` vs. `caja_danada`) y una interfaz web en **Streamlit** con API en **Flask**.

### Arquitectura de Componentes (Avance 1 a Proyecto Final)

La solución está estructurada modularmente bajo el principio de inversión de dependencias (`BaseIntentClassifier`), lo cual permite desacoplar el gestor de diálogo de la técnica de clasificación (reglas simbólicas vs. Machine Learning):

```
┌─────────────────────────────────────────────────────────────┐
│                      Inspector (Usuario)                     │
└──────────────────────────────┬──────────────────────────────┘
                               │ Texto natural
                               ▼
┌─────────────────────────────────────────────────────────────┐
│               DialogueManager (Gestor de Diálogo)            │
│  - Enrutamiento conversacional, slots pendientes y fallback  │
└──────────┬───────────────────┬───────────────────┬──────────┘
           │                   │                   │
           ▼                   ▼                   ▼
┌─────────────────────┐ ┌────────────────┐ ┌───────────────────┐
│ BaseIntentClassifier│ │ EntityExtractor│ │  AuditoriaSession │
│ (RuleBased/ML TFIDF)│ │ (Lote, Andén)  │ │ (Estado y Reglas) │
└──────────┬──────────┘ └────────────────┘ └─────────┬─────────┘
           │                                         │
           ▼                                         ▼
┌─────────────────────┐                     ┌───────────────────┐
│  data/intents.json  │                     │    bitacora.json  │
└─────────────────────┘                     └───────────────────┘
```

### Módulo 1: Preprocesamiento de Lenguaje Natural (NLTK)
Aplica técnicas formales de limpieza, normalización NFD (remoción de acentos), tokenización léxica (`wordpunct_tokenize`) y reducción morfológica por stemming para español (`SnowballStemmer`).

In [1]:
from src.preprocessor import normalizar, obtener_raiz, tokenizar_y_stem

ejemplo_texto = "¡Alerta! Las cajas-llegaron dañadas, perforadas y mojadas en el andén 2."
print("Texto original :", ejemplo_texto)
print("Normalizado    :", normalizar(ejemplo_texto))
print("Tokens + Stems :", tokenizar_y_stem(ejemplo_texto))

Texto original : ¡Alerta! Las cajas-llegaron dañadas, perforadas y mojadas en el andén 2.
Normalizado    : alerta las cajas llegaron danadas perforadas y mojadas en el anden 2
Tokens + Stems : ['alert', 'las', 'caj', 'lleg', 'dan', 'perfor', 'y', 'moj', 'en', 'el', 'anden', '2']


### Módulo 2: Extracción de Entidades Operativas (RegEx y Slot-Filling)
Extrae de forma robusta identificadores de lote (ej. `LOTE-504`, `710`) y andenes o bahías de recepción (ej. `andén 2`, `bahía 3`), soportando respuestas directas con números aislados para llenado de slots.

In [2]:
from src.entity_extractor import extraer_entidades, extraer_numero_aislado

msg_1 = "Voy a auditar el lote 504 en el andén 2"
msg_2 = "Revisando llegada de refacciones en bahía 3"
msg_3 = "3"

print(f"Mensaje: '{msg_1}' -> Entidades:", extraer_entidades(msg_1))
print(f"Mensaje: '{msg_2}' -> Entidades:", extraer_entidades(msg_2))
print(f"Mensaje: '{msg_3}' -> Número aislado (slot):", extraer_numero_aislado(msg_3))

Mensaje: 'Voy a auditar el lote 504 en el andén 2' -> Entidades: {'lote': '504', 'anden': 2}
Mensaje: 'Revisando llegada de refacciones en bahía 3' -> Entidades: {'lote': None, 'anden': 3}
Mensaje: '3' -> Número aislado (slot): 3


### 5. Módulo 3: Base de Conocimiento y Motor de Reglas (Chatbot Engine)
Carga el catálogo `data/intents.json` e indexa raíces de palabras y secuencias de frases compuestas. La interfaz `BaseIntentClassifier` garantiza compatibilidad directa con futuros clasificadores de Machine Learning.

In [7]:
from src.chatbot_engine import RuleBasedChatbot

bot = RuleBasedChatbot("data/intents.json")
print("Intenciones cargadas:", bot.obtener_intenciones())

# Pruebas de clasificación directa
frases_test = [
    "Hola, buen día",
    "Voy a auditar el lote 504 en el andén 2",
    "La caja viene rota de una esquina",
    "Las cajas vienen en perfecto estado",
    "¿Cuáles son los motivos para rechazar una caja?",
    "Quiero pedir comida"
]

for f in frases_test:
    tag = bot.clasificar(f)
    print(f"'{f}' -> [{tag}]")

Intenciones cargadas: ['saludo', 'iniciar_auditoria', 'reportar_dano_empaque', 'confirmar_empaque_intacto', 'consultar_politica_rechazo', 'despedida', 'fallback']
'Hola, buen día' -> [saludo]
'Voy a auditar el lote 504 en el andén 2' -> [iniciar_auditoria]
'La caja viene rota de una esquina' -> [reportar_dano_empaque]
'Las cajas vienen en perfecto estado' -> [confirmar_empaque_intacto]
'¿Cuáles son los motivos para rechazar una caja?' -> [consultar_politica_rechazo]
'Quiero pedir comida' -> [fallback]


### 6. Módulo 4: Gestor de Estado y Reglas de Negocio (Auditoría Session)
Controla el ciclo de vida de la auditoría, gestiona los identificadores (lote y andén/bahía), maneja la completitud de slots y dictamina el estatus de calidad del empaque (`RECHAZADO_CUARENTENA` vs. `APROBADO_LIBERADO`).

In [8]:
from src.auditoria_manager import AuditoriaSession

sesion = AuditoriaSession(responsable="Inspector_Anden")
sesion.iniciar_inspeccion(lote="LOTE-504", anden=24)
print("¿Identificadores completos? :", sesion.identificadores_completos)
print(f"Lote activo: {sesion.lote} | Andén activo: {sesion.anden}")

dictamen = sesion.evaluar_empaque(es_danado=True)
print("Dictamen emitido :", dictamen)
print("Resumen sesión   :", sesion.obtener_resumen_reporte())

¿Identificadores completos? : True
Lote activo: LOTE-504 | Andén activo: 24
Dictamen emitido : 🚨 Lote LOTE-504 RECHAZADO en Andén 24. Empaque con daños físicos. Segregar bulto en zona de cuarentena y solicitar firma de no-conformidad al transportista.
Resumen sesión   : {'responsable': 'Inspector_Anden', 'fecha': '2026-10-23 20:10:26', 'lote': 'LOTE-504', 'anden': 24, 'dictamen': 'RECHAZADO_CUARENTENA', 'en_progreso': True, 'total_eventos': 2}


### 7. Módulo 5: Orquestador Conversacional (Dialogue Manager)
Integra el clasificador, la extracción de entidades y el gestor de estado para coordinar el diálogo continuo, gestionar slots incompletos y emitir fallbacks contextualizados.

In [9]:
from src.dialogue_manager import DialogueManager

gestor = DialogueManager(classifier=bot, session=AuditoriaSession())

dialogo_demo = [
    "Buenas tardes, quiero iniciar turno",
    "Revisando llegada de refacciones en bahía 3",
    "lote 710",
    "La caja viene rota de una esquina",
    "Es todo por el momento"
]

for entrada in dialogo_demo:
    resp = gestor.responder(entrada)
    print(f"Inspector : {entrada}")
    print(f"Bot       : {resp}")
    print("-" * 60)

Inspector : Buenas tardes, quiero iniciar turno
Bot       : Hola. Soy el Asistente de Auditoría de Almacén. Puedo ayudarte a registrar inspecciones de carga, consultar criterios de empaque y generar reportes de no-conformidad. ¿Qué lote deseas revisar?
------------------------------------------------------------
Inspector : Revisando llegada de refacciones en bahía 3
Bot       : Entendido. ¿Cuál es el número de lote que vas a auditar?
------------------------------------------------------------
Inspector : lote 710
Bot       : Entendido. He registrado el inicio de auditoría del lote 710 en el andén 3. Por favor, realiza la inspección física del paquete y describe si el empaque presenta golpes o si se encuentra intacto.
------------------------------------------------------------
Inspector : La caja viene rota de una esquina
Bot       : [Lote 710 · andén 3] Alerta de daño registrada. De acuerdo con el protocolo de calidad, no introduzcas el paquete al inventario general. Por favor, segr

### 8. Módulo 6: Batería de Pruebas Automatizadas (10 Conversaciones Oficiales)
Ejecución de las 10 conversaciones definidas en el documento MHTML (38 turnos en total) evaluando precisión, tasa de fallback y exportando la bitácora transaccional a `data/bitacora_pruebas.json`.

In [10]:
from tests.run_tests import ejecutar_evaluacion

metricas = ejecutar_evaluacion(verbose=True)


 BATERÍA DE PRUEBAS FUNCIONALES - ASISTENTE DE AUDITORÍA DE ALMACÉN

--- C1 Flujo completo conforme ---
Usuario : Hola, buen día
Bot     : Hola. Soy el Asistente de Auditoría de Almacén. Puedo ayudarte a registrar inspecciones de carga, consultar criterios de empaque y generar reportes de no-conformidad. ¿Qué lote deseas revisar?
          [Esperada: saludo                     | Detectada: saludo                    ] [OK]
Usuario : Voy a auditar el lote 504 en el andén 2
Bot     : Entendido. He registrado el inicio de auditoría del lote 504 en el andén 2. Por favor, realiza la inspección física del paquete y describe si el empaque presenta golpes o si se encuentra intacto.
          [Esperada: iniciar_auditoria          | Detectada: iniciar_auditoria         ] [OK]
Usuario : Las cajas vienen en perfecto estado
Bot     : [Lote 504 · andén 2] Excelente. Inspección de empaque aprobada. El material ha sido validado para su liberación e ingreso al almacén general. Puedes proceder con la es

### 9. Pruebas de Límites y Justificación Técnica para Avance 2 (Machine Learning)

| Caso de Límite | Entrada del Inspector | Intención Deseada | Comportamiento en Reglas | Solución Técnica en Avance 2 (TF-IDF + SVM) |
| :--- | :--- | :--- | :--- | :--- |
| **Errata / Typo** | *"la caja esta aplastda"* | `reportar_dano_empaque` | Falla o cae en fallback si la errata no está en el diccionario estático. | N-gramas de caracteres en TF-IDF capturan similitud sub-léxica tolerante a erratas. |
| **Sinónimo no registrado** | *"la caja trae un hueco enorme"* | `reportar_dano_empaque` | Coincidencia nula al no figurar *"hueco"* en keywords. | Embeddings léxicos o pesos TF-IDF en corpus supervisado generalizan patrones semánticos. |
| **Variación léxica** | *"el bulto viene en mal estado"* | `reportar_dano_empaque` | Reglas estrictas ignoran *"mal estado"* si solo se programó *"dañado"*. | Modelo SVM con regularización lineal separa hiperplanos basados en co-ocurrencia. |
| **Ambigüedad** | *"hay una caja pero no sé si está bien"* | `fallback` | Puede disparar falsos positivos si contiene palabras aisladas como *"caja"*. | Clasificación probabilística con umbral de incertidumbre (confianza < 0.65 -> fallback asistido). |

### 10. Conclusiones y Hoja de Ruta hacia las Siguientes Fases

1. **Cumplimiento Integral Avance 1:**
   - Se implementó exitosamente el preprocesamiento con Python y **NLTK** (tokenización y stemming en español).
   - Se estructuró la base de conocimiento en `intents.json` con catálogo ampliado y soporte de frases clave compuestas.
   - Se garantizó la extracción robusta de entidades operativas (lote y andén) y el llenado contextual de datos (*slot-filling*).
   - Se validaron las reglas de negocio (andenes numéricos sin restricción de rango) y se generó la bitácora transaccional en JSON para trazabilidad.
   - Se alcanzó un **100% de precisión en los 38 turnos de las 10 conversaciones oficiales**.

2. **Preparación para Avance 2 (NLP Supervisado):**
   - Gracias a la clase abstracta `BaseIntentClassifier`, se integrará `MLIntentClassifier(BaseIntentClassifier)` utilizando `TfidfVectorizer` + `LinearSVC` entrenado con un dataset enriquecido, superando las limitaciones de vocabulario cerrado.

3. **Preparación para Proyecto Final (Visión Multimodal + Flask + Streamlit):**
   - La sesión de auditoría (`AuditoriaSession`) se vinculará con un clasificador de visión artificial binario (**ResNet50**) para clasificar fotos de empaques (`caja_intacta` vs. `caja_danada`).
   - La lógica desacoplada en `src/` será expuesta mediante una API REST en **Flask** y consumida por una interfaz web interactiva en **Streamlit**.